# 2. Preprocessing and Feature Engineering

This notebook prepares the customer information dataset for clustering. The reusable preprocessing functions are stored in `utils/preprocessing_functions.py`.

The notebook is organised into four stages:

1. Data cleaning
2. Feature engineering
3. Missing-value imputation
4. Validation and export

## Imports and Data

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)

In [2]:
import pandas as pd

from utils.preprocessing_functions import (
    fix_promotion_percentage,
    correct_first_transaction_year,
    create_age_feature,
    extract_degree_feature,
    create_loyalty_card_feature,
    create_complaint_feature,
    create_children_feature,
    encode_gender,
    create_time_category_feature,
    create_customer_lifetime_feature,
    create_annual_spend_features,
    impute_distinct_stores,
    evaluate_vegetable_imputer,
    impute_vegetables_with_boosting,
    impute_spend_with_knn,
    validate_customer_data,
)

In [3]:
customer_info = pd.read_csv(r'C:\Users\nhatp\OneDrive - NOVAIMS\gits\customer-segmentation\data\raw\customer_info.csv')

customer_info = customer_info.drop(columns="Unnamed: 0")

customer_info.head()

,customer_id,customer_name,customer_gender,customer_birthdate,kids_home,teens_home,number_complaints,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,...,lifetime_spend_fish,lifetime_spend_hygiene,lifetime_spend_videogames,lifetime_spend_petfood,lifetime_total_distinct_products,percentage_of_products_bought_promotion,year_first_transaction,loyalty_card_number,latitude,longitude
0,29930,April Clark,female,01/15/1972 02:27 PM,2.0,2.0,1.0,4.0,7789.0,5601.0,...,1894.0,457.0,412.0,428.0,386.0,0.158741,2018.0,NaN,38.721807,-9.125534
1,6813,Bsc. Paul Ketchum,male,07/31/1944 10:53 AM,0.0,1.0,0.0,4.0,8653.0,35.0,...,346.0,394.0,75.0,226.0,73.0,1.227890,2013.0,971840.0,38.734668,-9.163533
2,39451,Mary Downing,female,11/13/1989 02:11 PM,2.0,3.0,0.0,7.0,15605.0,4275.0,...,1971.0,920.0,335.0,192.0,319.0,0.101598,2011.0,NaN,38.787126,-9.147077
3,21557,Manuel Kueny,male,08/09/1976 06:23 AM,0.0,0.0,1.0,1.0,13440.0,16366.0,...,785.0,139.0,679.0,270.0,221.0,0.259943,2009.0,NaN,38.741816,-9.159700
4,16415,Phd. Curtis Tharp,male,07/11/1966 08:12 AM,1.0,1.0,1.0,5.0,49250.0,3197.0,...,1492.0,1046.0,112.0,144.0,244.0,0.317822,2012.0,925367.0,38.785921,-9.149221


## Initial Data Quality

In [4]:
print(f'Customers: {customer_info.shape[0]}')
print(f'Features: {customer_info.shape[1]}')

Customers: 34060
Features: 25


In [5]:
customer_info.isna().sum().sort_values(ascending=False)

loyalty_card_number                        10908
lifetime_spend_fish                         1703
typical_hour                                1362
lifetime_spend_vegetables                   1022
number_complaints                           1022
teens_home                                   783
kids_home                                    749
distinct_stores_visited                      681
lifetime_spend_alcohol_drinks                681
customer_birthdate                           341
customer_name                                  0
customer_gender                                0
customer_id                                    0
lifetime_spend_groceries                       0
lifetime_spend_nonalcohol_drinks               0
lifetime_spend_meat                            0
lifetime_spend_electronics                     0
lifetime_spend_hygiene                         0
lifetime_spend_videogames                      0
lifetime_total_distinct_products               0
lifetime_spend_petfo

In [6]:
customer_info['percentage_of_products_bought_promotion'].describe()

count    34060.000000
mean         0.337093
std          0.322766
min         -1.340710
25%          0.113353
50%          0.240373
75%          0.514691
max          1.952971
Name: percentage_of_products_bought_promotion, dtype: float64

## 1. Data Cleaning

Promotion percentages below zero are treated as sign errors. Values between 1 and 10 are treated as decimal-place errors. Values above 10 cannot be interpreted reliably and are set to missing.

First-transaction years after the reference year are corrected to avoid negative customer lifetimes.

In [7]:
customer_info = fix_promotion_percentage(customer_info)
customer_info = correct_first_transaction_year(customer_info)

customer_info['pct_prod_in_promotion'].describe()

count    34060.000000
mean         0.303255
std          0.249096
min          0.000006
25%          0.112916
50%          0.218856
75%          0.439476
max          0.999797
Name: pct_prod_in_promotion, dtype: float64

## 2. Feature Engineering

The raw demographic and behavioural variables are converted into modelling features. Missing loyalty-card numbers, complaints, and children at home are interpreted according to their business meaning.

In [8]:
customer_info = create_age_feature(customer_info)
customer_info = extract_degree_feature(customer_info)
customer_info = create_loyalty_card_feature(customer_info)
customer_info = create_complaint_feature(customer_info)
customer_info = create_children_feature(customer_info)
customer_info = encode_gender(customer_info)
customer_info = create_time_category_feature(customer_info)

customer_info.head()

,customer_id,distinct_stores_visited,lifetime_spend_groceries,lifetime_spend_electronics,lifetime_spend_vegetables,lifetime_spend_nonalcohol_drinks,lifetime_spend_alcohol_drinks,lifetime_spend_meat,lifetime_spend_fish,lifetime_spend_hygiene,...,year_first_transaction,latitude,longitude,age,degree,loyalty_card,complained,child_home,male,typical_time
0,29930,4.0,7789.0,5601.0,726.0,962.0,1213.0,1598.0,1894.0,457.0,...,2018.0,38.721807,-9.125534,53,0,0,1,4.0,0,1
1,6813,4.0,8653.0,35.0,792.0,102.0,104.0,741.0,346.0,394.0,...,2013.0,38.734668,-9.163533,81,1,1,0,1.0,1,1
2,39451,7.0,15605.0,4275.0,1585.0,980.0,1872.0,1323.0,1971.0,920.0,...,2011.0,38.787126,-9.147077,36,0,0,0,5.0,0,1
3,21557,1.0,13440.0,16366.0,28.0,269.0,1855.0,939.0,785.0,139.0,...,2009.0,38.741816,-9.159700,49,0,0,1,0.0,1,1
4,16415,5.0,49250.0,3197.0,258.0,726.0,547.0,983.0,1492.0,1046.0,...,2012.0,38.785921,-9.149221,59,1,1,1,2.0,1,1


### Customer Lifetime and Annual Spending

Lifetime spending is divided by the number of years since the customer's first transaction. This makes customers with different relationship lengths comparable.

If a customer has zero or negative lifetime, the annual spending is set to missing and handled during imputation.

In [9]:
customer_info = create_customer_lifetime_feature(customer_info)
customer_info = create_annual_spend_features(customer_info)

customer_info[[
    'customer_lifetime',
    'lifetime_spend_groceries',
    'spend_per_year_groceries',
    'lifetime_spend_electronics',
    'spend_per_year_electronics',
]].head()

,customer_lifetime,lifetime_spend_groceries,spend_per_year_groceries,lifetime_spend_electronics,spend_per_year_electronics
0,7.0,7789.0,1112.714286,5601.0,800.142857
1,12.0,8653.0,721.083333,35.0,2.916667
2,14.0,15605.0,1114.642857,4275.0,305.357143
3,16.0,13440.0,840.000000,16366.0,1022.875000
4,13.0,49250.0,3788.461538,3197.0,245.923077


## 3. Missing-Value Imputation

Vegetable spending has weaker correlations with the other spending categories, so it is imputed with a gradient-boosting regression model. The remaining annual spending variables are imputed with K-Nearest Neighbours.

Before imputing, the vegetable model is evaluated on a hold-out sample of customers with known vegetable spending.

In [10]:
evaluation = evaluate_vegetable_imputer(customer_info)

evaluation

{'mse': np.float64(3460.9297227242632), 'r2': np.float64(0.7552649934778222)}

In [11]:
customer_info = impute_vegetables_with_boosting(customer_info)
customer_info = impute_spend_with_knn(customer_info, target_columns=None)

In [12]:
customer_info.filter(like='spend_per_year_').isna().sum()

spend_per_year_groceries            0
spend_per_year_electronics          0
spend_per_year_vegetables           0
spend_per_year_nonalcohol_drinks    0
spend_per_year_alcohol_drinks       0
spend_per_year_meat                 0
spend_per_year_fish                 0
spend_per_year_hygiene              0
spend_per_year_petfood              0
spend_per_year_videogames           0
dtype: int64

In [13]:
customer_info = impute_distinct_stores(customer_info)

In [ ]:
lifetime_columns = [
    column for column in customer_info.columns
    if column.startswith("lifetime_spend_")
]

customer_info_clean = customer_info.drop(columns=lifetime_columns)

In [ ]:
customer_info_clean.isna().sum().sort_values(ascending=False)

customer_id                         0
distinct_stores_visited             0
lifetime_total_distinct_products    0
pct_prod_in_promotion               0
year_first_transaction              0
latitude                            0
longitude                           0
age                                 0
degree                              0
loyalty_card                        0
complained                          0
child_home                          0
male                                0
typical_time                        0
customer_lifetime                   0
spend_per_year_groceries            0
spend_per_year_electronics          0
spend_per_year_vegetables           0
spend_per_year_nonalcohol_drinks    0
spend_per_year_alcohol_drinks       0
spend_per_year_meat                 0
spend_per_year_fish                 0
spend_per_year_hygiene              0
spend_per_year_petfood              0
spend_per_year_videogames           0
dtype: int64

## 4. Validation and Export

The final dataset is checked before it is used for clustering. The checks confirm that customer identifiers are unique, promotion percentages are valid, customer lifetimes are positive, and annual spending has no missing values.

In [ ]:
validation_results = validate_customer_data(customer_info_clean)

validation_results

unique_customer_id            True
promotion_within_range        True
positive_customer_lifetime    True
no_missing_annual_spend       True
Name: passed, dtype: bool

In [ ]:
customer_info_clean.to_csv(r'C:\Users\nhatp\OneDrive - NOVAIMS\gits\customer-segmentation\data\preprocessed\customer_info_clean.csv', index=False)

print(f'Final dataset shape: {customer_info_clean.shape}')

Final dataset shape: (34060, 25)


In [ ]:
customer_info_clean.head()

,customer_id,distinct_stores_visited,lifetime_total_distinct_products,pct_prod_in_promotion,year_first_transaction,latitude,longitude,age,degree,loyalty_card,...,spend_per_year_groceries,spend_per_year_electronics,spend_per_year_vegetables,spend_per_year_nonalcohol_drinks,spend_per_year_alcohol_drinks,spend_per_year_meat,spend_per_year_fish,spend_per_year_hygiene,spend_per_year_petfood,spend_per_year_videogames
0,29930,4,386.0,0.158741,2018.0,38.721807,-9.125534,53,0,0,...,1112.714286,800.142857,103.714286,137.428571,173.285714,228.285714,270.571429,65.285714,61.142857,58.857143
1,6813,4,73.0,0.122789,2013.0,38.734668,-9.163533,81,1,1,...,721.083333,2.916667,66.000000,8.500000,8.666667,61.750000,28.833333,32.833333,18.833333,6.250000
2,39451,7,319.0,0.101598,2011.0,38.787126,-9.147077,36,0,0,...,1114.642857,305.357143,113.214286,70.000000,133.714286,94.500000,140.785714,65.714286,13.714286,23.928571
3,21557,1,221.0,0.259943,2009.0,38.741816,-9.159700,49,0,0,...,840.000000,1022.875000,1.750000,16.812500,115.937500,58.687500,49.062500,8.687500,16.875000,42.437500
4,16415,5,244.0,0.317822,2012.0,38.785921,-9.149221,59,1,1,...,3788.461538,245.923077,19.846154,55.846154,42.076923,75.615385,114.769231,80.461538,11.076923,8.615385


## Preprocessing Decisions

- Missing loyalty-card numbers are interpreted as customers without a loyalty card.
- Missing complaint values are interpreted as no registered complaint.
- Missing children and teenagers at home are interpreted as zero dependants.
- Missing typical shopping hours are imputed with the most frequent value.
- Missing age values are imputed with the median age.
- Vegetable spending is imputed using a gradient-boosting regression model.
- Remaining annual spending values are imputed using K-Nearest Neighbours.
- Lifetime spending is converted into annual spending to account for different customer lifetimes.